# Car Detection in Snow – Project Tutorial
**Course:** D7047E Advanced Deep Learning | LTU VT2026  
**Dataset:** [Nordic Vehicle Dataset (NVD)](https://nvd.ltu-ai.dev/)  
**Models:** YOLOv9 (ultralytics), DETR (HuggingFace), Faster R-CNN (torchvision)

---

## What is this project?

Autonomous vehicles and surveillance systems must operate reliably in **adverse weather conditions**. Snow introduces:
- **Occlusion** – vehicles hidden behind snowbanks or snow-covered objects
- **Low contrast** – white vehicles blending into white backgrounds
- **Texture homogenisation** – road markings, edges, and details disappear under snow

This project **fine-tunes and compares** three deep learning object detectors on the **Nordic Vehicle Dataset (NVD)** – a UAV (drone) dataset of aerial images captured over northern Sweden under real winter conditions. We also investigate whether **synthetic snow augmentation** during training improves detection performance.

### Research Questions
1. How much does fine-tuning on winter data improve over a COCO-pretrained zero-shot baseline?
2. Does adding synthetic snow augmentation during training improve performance on snowy test frames?
3. How do one-stage (YOLOv9) vs. two-stage (Faster R-CNN) vs. transformer-based (DETR) detectors compare on this winter domain?

---

## Tutorial Contents

| Section | What you learn |
|---------|---------------|
| [1. Setup & Project Structure](#1-setup--project-structure) | Install dependencies, understand folder layout |
| [2. Dataset Overview](#2-dataset-overview) | What NVD is, how it is structured |
| [3. Loading the Dataset](#3-loading-the-dataset) | Using `data_utils` to parse YOLO annotations |
| [4. Dataset Statistics](#4-dataset-statistics) | Count images, boxes, classes per split |
| [5. Annotation Quality Check](#5-annotation-quality-check) | Detect crowd boxes, tiny boxes, empty frames |
| [6. Exploratory Data Analysis](#6-exploratory-data-analysis) | Visualise images, distributions, heatmaps |
| [7. Snow Severity Labelling](#7-snow-severity-labelling) | Manually label frames by snow intensity |
| [8. Dataset Preparation](#8-dataset-preparation) | Verify annotations, convert to COCO for DETR |
| [9. Model Training – YOLOv9](#9-model-training--yolov9) | Fine-tune YOLOv9 with ultralytics |
| [10. Model Training – DETR](#10-model-training--detr) | Fine-tune DETR with HuggingFace Transformers |
| [11. Evaluation & Results](#11-evaluation--results) | Compute mAP, precision, recall, FPS |
| [12. Error Analysis](#12-error-analysis) | Understand where models fail |

---
## 1. Setup & Project Structure

### Installation

```bash
pip install -r requirements.txt
```

> **GPU Note:** `requirements.txt` is configured for CUDA 12.4 (`torch>=2.3.0+cu124`).
> If you are on CPU only, replace the `--extra-index-url` line and install `torch` without the CUDA suffix.

### Project Structure

```
Project/
├── configs/
│   ├── data.yaml          ← YOLO dataset config — pass directly to ultralytics
│   └── splits.yaml        ← recording-level split config (edit paths after download)
├── data/
│   ├── raw/               ← NVD download goes here (images/ + labels/ + *.txt split files)
│   └── processed/         ← COCO JSON files written here by prepare_coco_format.py
├── models/                ← saved model checkpoints
├── results/
│   └── figures/           ← plots and visualisation outputs
├── scripts/
│   ├── data_utils/        ← Python package: data model, loaders, stats, formatter
│   │   ├── models.py           ← domain dataclasses: BBox, ImageAnnotation, DatasetSplit
│   │   ├── loaders.py          ← YOLO file parsing: .txt labels → absolute pixel BBoxes
│   │   ├── config_loader.py    ← YAML loading: data.yaml / splits.yaml → DatasetSplit list
│   │   ├── dataset_stats.py    ← pure computation: compute_stats(split) → dict
│   │   ├── reporting.py        ← presentation only: print_stats(stats) → stdout
│   │   └── coco_formatter.py   ← serialisation: DatasetSplit → COCO JSON
│   ├── eda/               ← Python package: bounding-box visualiser, distribution plots
│   ├── inspect_dataset.py      ← CLI: print per-split statistics
│   ├── verify_annotations.py   ← CLI: draw boxes on sampled images and save
│   ├── label_snow_severity.py  ← CLI: interactive frame labeller (light/medium/heavy)
│   └── prepare_coco_format.py  ← CLI: convert YOLO splits to COCO JSON
├── main.ipynb             ← primary experiment notebook
├── tutorial.ipynb         ← this file
├── requirements.txt
├── EXPERIMENTS.md         ← training run log
└── TODO_CarDetection_Snow.md  ← project checklist
```

### Why so many files in `data_utils`?

This follows a design principle called **Separation of Concerns (SoC)** — each file has exactly one job, so you can read, test, and change it without touching anything else:

| Module | Single responsibility |
|--------|----------------------|
| `models.py` | Define *what* the data looks like — the `BBox`, `ImageAnnotation`, `DatasetSplit` dataclasses |
| `loaders.py` | *Read* YOLO `.txt` files from disk and convert normalised coordinates to absolute pixels |
| `config_loader.py` | *Interpret* YAML config files and call the right loader |
| `dataset_stats.py` | *Compute* numerical summaries — pure functions, no I/O side effects |
| `reporting.py` | *Display* stats as human-readable text — no computation, just formatting |
| `coco_formatter.py` | *Serialise* parsed data to COCO JSON for transformer-based models |

> **Why does this matter?**
> If you only need to fix a bug in statistics computation, you open `dataset_stats.py` and nothing else.
> If you want to change how results are printed, you only touch `reporting.py`.
> This prevents the classic beginner mistake of putting everything in one giant file.

---
## 1b. Key Concepts for Beginners

Before diving into code, here are the core ideas you need to understand.

---

### Concept 1: The Data Pipeline

Data flows through the project in a clear sequence. Each step adds structure:

```
Raw files on disk
    │
    ▼  loaders.py
In-memory Python objects  (BBox, ImageAnnotation, DatasetSplit)
    │
    ├──▶  dataset_stats.py  →  numbers (counts, averages, distributions)
    │         │
    │         └──▶  reporting.py  →  printed table on screen
    │
    ├──▶  eda/visualizer.py  →  matplotlib figures
    │
    └──▶  coco_formatter.py  →  COCO JSON file on disk
```

The key insight: **parse once, use many ways**. You load the dataset into `DatasetSplit` objects once, and then pass those same objects to statistics, visualisation, and export functions – no re-reading files.

---

### Concept 2: YOLO Annotation Format

Every image in the dataset has a companion `.txt` file. Each line in that file describes one bounding box:

```
class_id  centre_x  centre_y  width  height
```

All five numbers are **normalised to [0, 1]** relative to the image size. For example:
```
0  0.512  0.441  0.183  0.210
```
Means: **class 0** (car), box centred at **51.2% across**, **44.1% down** the frame, with box **18.3% wide** and **21.0% tall**.

The code in `loaders.py` converts these back to **absolute pixel coordinates** (e.g. `x_min = 312px`) so every downstream module works in a single coordinate system.

---

### Concept 3: Dataclasses

The project uses Python `@dataclass` to define its data structures. A dataclass is just a class that holds data with minimal boilerplate:

```python
@dataclass
class BBox:
    class_id: int          # which class (0 = car)
    class_name: str        # human label ("car")
    x_min: float           # left edge in pixels
    y_min: float           # top edge in pixels
    x_max: float           # right edge in pixels
    y_max: float           # bottom edge in pixels

    @property
    def width(self) -> float:
        return self.x_max - self.x_min   # computed on the fly, not stored
```

Think of `BBox` as a named container for one bounding box. `ImageAnnotation` holds one image and its list of `BBox` objects. `DatasetSplit` holds a whole split (train/val/test) and its list of `ImageAnnotation` objects.

---

### Concept 4: Separation of Concerns

A common mistake in ML projects is writing a 1000-line script that loads data, computes stats, prints them, saves plots, and exports formats – all in the same function. This is hard to debug and impossible to reuse.

The **Separation of Concerns** principle says: *one module, one job*.

| Bad (tangled) | Good (separated) |
|---------------|-----------------|
| `load_and_print_stats()` | `compute_stats()` in one file, `print_stats()` in another |
| Parser also writes COCO JSON | Parser returns objects; formatter writes JSON |
| CLI script duplicates loading logic | All scripts call the same shared `load_splits_from_data_yaml()` |

You will see this pattern throughout the codebase.

In [ ]:

# -- Section 1: Environment check --------------------------------------------
import sys
import importlib

print(f"Python {sys.version}")

required = ["torch", "torchvision", "ultralytics", "cv2", "PIL", "yaml",
            "matplotlib", "seaborn", "pandas", "numpy", "tqdm"]

missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
        print(f"  [OK]  {pkg}")
    except ImportError:
        print(f"  [MISSING]  {pkg}  <- NOT installed")
        missing.append(pkg)

if missing:
    print(f"\n[!] Missing packages: {missing}")
    print("Run:  pip install -r requirements.txt")
else:
    print("\nAll dependencies are installed.")


In [ ]:

# -- Section 1: Common imports and project paths ------------------------------
import os
import random
import sys
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Always run from the project root so relative paths work
PROJECT_ROOT = Path(__file__).resolve().parent if "__file__" in dir() else Path(".").resolve()
os.chdir(PROJECT_ROOT)

# Add scripts/ to sys.path so data_utils and eda packages are importable
sys.path.insert(0, str(PROJECT_ROOT / "scripts"))

# Key directories
DATA_RAW   = PROJECT_ROOT / "data" / "raw"
DATA_PROC  = PROJECT_ROOT / "data" / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
FIGURES    = PROJECT_ROOT / "results" / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

# Reproducibility
random.seed(42)
np.random.seed(42)

print("Working directory:", PROJECT_ROOT)
print("data/raw exists:", DATA_RAW.exists())


---
## 2. Dataset Overview

The **Nordic Vehicle Dataset (NVD)** is a UAV (drone) dataset – frames from aerial video captured by UAVs over northern Sweden.

| Property | Details |
|----------|---------|
| Source | [nvd.ltu-ai.dev](https://nvd.ltu-ai.dev/) (free, requires registration) |
| Capture device | UAV / drone (oblique aerial view) |
| Conditions | Winter – snow, ice, low-light, fog |
| Annotation format | **YOLO format** – one `.txt` per image, each line: `class_id cx cy w h` (normalised 0–1) |
| Classes | Vehicle(s) – confirmed `car` (class 0); verify your download for additional classes |

### How annotations work in YOLO format

Each image `foo.jpg` has a companion `foo.txt` where each line is:
```
class_id  centre_x  centre_y  width  height
```
All values are **normalised to the image dimensions** (0–1 range). For example:
```
0  0.512  0.441  0.183  0.210
```
means: class 0 (car), centred at (51.2%, 44.1%) of the frame, with box width 18.3% and height 21.0%.

### Dataset download and extraction

1. Register and download **"Labeled Frames (YOLO Format)"** from [nvd.ltu-ai.dev](https://nvd.ltu-ai.dev/)
2. Extract into `data/raw/` so the layout is:
   ```
   data/raw/
   ├── images/           ← all images (or per-recording subfolders)
   ├── labels/           ← matching .txt files
   ├── train.txt         ← list of training image paths
   ├── val.txt
   └── test.txt
   ```
3. Update `configs/splits.yaml` with the real recording subfolder names if the download ships multiple recordings separately.

In [ ]:

# -- Section 2: Peek at a raw label file ------------------------------------
# Shows you exactly what a YOLO annotation file looks like.

labels_dir = DATA_RAW / "labels"
label_files = list(labels_dir.glob("*.txt"))

if label_files:
    sample_label = label_files[0]
    print(f"Label file: {sample_label.name}\n")
    print("Contents:")
    print("-" * 40)
    print(sample_label.read_text().strip())
    print("-" * 40)
    print("\nFormat: class_id  cx  cy  width  height  (all normalised 0-1)")
else:
    print("No label files found in data/raw/labels/")
    print("Download and extract the NVD dataset first.")


---
## 3. Loading the Dataset

The `scripts/data_utils/` package provides a clean Python API to load the NVD dataset into memory.

### The three domain model classes (`models.py`)

```
DatasetSplit               ← one train/val/test split
  ├── name          : str            – "train", "val", or "test"
  ├── class_names   : list[str]      – ["car", ...]
  └── images        : list[ImageAnnotation]
        ├── file_name  : str
        ├── width, height : int      – pixels (read from image header)
        └── bboxes    : list[BBox]
              ├── class_id, class_name
              ├── x_min, y_min, x_max, y_max  ← absolute pixels
              ├── width, height, area          ← computed properties
              └── aspect_ratio
```

These are plain Python `@dataclass` objects – no framework magic. You can inspect, iterate, and pass them to any function.

### How loading works under the hood

1. **`config_loader.py`** reads `configs/data.yaml` to find the split `.txt` files and class names.
2. **`loaders.py`** opens each image path listed in those `.txt` files, reads the paired `.txt` label file, and converts every normalised `(cx, cy, w, h)` line into an absolute-pixel `BBox`.
3. The result is a `DatasetSplit` object you can work with immediately.

### Two ways to load splits

**Option A – `load_splits_from_data_yaml`** *(recommended – one call does everything)*
```python
from data_utils import load_splits_from_data_yaml

splits = load_splits_from_data_yaml("configs/data.yaml")
# Returns a list of DatasetSplit – one per split found in the YAML
```

**Option B – `load_splits_from_config`** *(when recordings are in separate subfolders)*
```python
from data_utils import load_splits_from_config

splits = load_splits_from_config("configs/splits.yaml")
```

> **Coordinate space:** Annotations are stored in YOLO format as normalised fractions.
> `loaders.py` converts them back to **absolute pixels** so all downstream code
> (stats, visualisation, COCO export) works in a single consistent coordinate system.
> You never need to multiply by image width/height yourself.

In [ ]:

# -- Section 3: Load all splits from data.yaml (Option A - recommended) ------
# load_splits_from_data_yaml() reads configs/data.yaml, finds the train/val/test
# .txt split files, opens each listed image, parses its YOLO label, and returns
# a list of DatasetSplit objects – all in one call.

from data_utils import load_splits_from_data_yaml

DATA_YAML = PROJECT_ROOT / "configs" / "data.yaml"

all_splits_list = load_splits_from_data_yaml(str(DATA_YAML))

splits = {}
for sp in all_splits_list:
    splits[sp.name] = sp
    print(f"  Loaded '{sp.name}': {sp.num_images} images, {sp.num_boxes} boxes")

if not splits:
    print("  No splits found – download the NVD dataset and update configs/data.yaml first.")

train_split = splits.get("train")
val_split   = splits.get("val")
test_split  = splits.get("test")


In [ ]:

# -- Section 3 (alt): Load from configs/splits.yaml (Option B) ---------------
# Use this when your NVD recordings are in separate per-recording subfolders.
# Edit configs/splits.yaml to point at the real folder names first.

# from data_utils import load_splits_from_config
#
# SPLITS_CONFIG = PROJECT_ROOT / "configs" / "splits.yaml"
# all_splits = load_splits_from_config(str(SPLITS_CONFIG))
#
# for sp in all_splits:
#     print(f"  {sp.name}: {sp.num_images} images, {sp.num_boxes} boxes")
#     print(f"  Classes: {sp.class_names}")

# -- Inspect one annotation manually -----------------------------------------
if "train_split" in dir() and train_split and train_split.images:
    first_img = train_split.images[0]
    print(f"Image:  {first_img.file_name}")
    print(f"Size:   {first_img.width} x {first_img.height} px")
    print(f"Boxes:  {first_img.num_boxes}")
    for i, bbox in enumerate(first_img.bboxes[:3]):
        print(f"  Box {i}: {bbox.class_name}  "
              f"[{bbox.x_min:.0f}, {bbox.y_min:.0f}, {bbox.x_max:.0f}, {bbox.y_max:.0f}]  "
              f"area={bbox.area:.0f}px2  ar={bbox.aspect_ratio:.2f}")
else:
    print("No splits loaded – run the cell above after downloading the dataset.")


---
## 4. Dataset Statistics

### Two separate functions – why?

This is Separation of Concerns in action:

| Function | Module | Job |
|----------|--------|-----|
| `compute_stats(split)` | `dataset_stats.py` | **Pure computation** – takes a `DatasetSplit`, returns a plain `dict` of numbers and lists. No printing, no I/O. |
| `print_stats(stats)` | `reporting.py` | **Pure presentation** – takes that `dict` and formats it for the terminal. No computation. |

Keeping them separate means you can call `compute_stats()` in a loop to collect results without printing, then choose *when and how* to display them. It also makes testing easy – you can check the numbers without capturing printed output.

### What `compute_stats` returns

| Key | Type | Description |
|-----|------|-------------|
| `num_images` | int | Total images in split |
| `num_boxes` | int | Total annotated boxes |
| `class_counts` | dict | Boxes per class |
| `images_without_boxes` | int | Empty frames (no annotations) |
| `boxes_per_image` | list[int] | Distribution of object counts per frame |
| `box_widths` / `box_heights` | list[float] | Absolute pixel dimensions |
| `box_areas` | list[float] | Width x height (px²) |
| `box_aspect_ratios` | list[float] | Width / height |
| `bbox_centers_x/y` | list[float] | Normalised [0, 1] centre positions |
| `crowd_boxes` | int | Boxes with `iscrowd=1` (groups, not individually annotated) |
| `tiny_boxes` | int | Boxes with area < 32×32 px (edge objects or label noise) |
| `missing_dims` | int | Images where width/height was unreadable |

The equivalent **CLI command** is:
```bash
python scripts/inspect_dataset.py --data-yaml configs/data.yaml
```

In [ ]:

# -- Section 4: Compute and print stats for all splits -----------------------
from data_utils import compute_stats
from data_utils.reporting import print_stats  # presentation layer, separate from computation

if splits:
    all_stats = {}
    for name, sp in splits.items():
        s = compute_stats(sp)
        all_stats[name] = s
        print_stats(s)
else:
    print("No splits loaded – run the data loading cells in Section 3 first.")


In [ ]:

# -- Section 4: Summary table across splits ----------------------------------
# Builds a side-by-side pandas table so you can quickly compare splits.

if all_stats:
    rows = []
    for name, s in all_stats.items():
        bpi = s["boxes_per_image"]
        rows.append({
            "Split": name,
            "Images": s["num_images"],
            "Boxes": s["num_boxes"],
            "Avg boxes/img": round(sum(bpi) / len(bpi), 2) if bpi else 0,
            "Empty frames": s["images_without_boxes"],
            "Tiny boxes": s["tiny_boxes"],
            "Crowd boxes": s["crowd_boxes"],
        })
    df = pd.DataFrame(rows).set_index("Split")
    display(df)
else:
    print("No stats available – run Section 3 first.")


---
## 5. Annotation Quality Check

Before training it is important to flag potential quality issues:

| Issue | What it means | How to handle |
|-------|--------------|---------------|
| **Crowd boxes** (`iscrowd=1`) | A group of objects annotated as one box | Use `iscrowd` flag during evaluation; exclude from instance-level mAP |
| **Tiny boxes** (area < 32×32 px) | Truncated or far-away vehicles; often label noise | Consider filtering with a minimum area threshold at training time |
| **Empty frames** | Images with no annotation | Expected in some sequences; verify they are truly vehicle-free |
| **Missing dims** | PIL could not read the image dimensions | Check for corrupt image files |

These numbers should be documented in your **Methodology** section in the report.

In [ ]:

# -- Section 5: Annotation quality report ------------------------------------
if all_stats:
    print(f"{'Split':<10} {'Crowd':>8} {'Tiny':>8} {'Empty frames':>14} {'Missing dims':>14}")
    print("-" * 56)
    for name, s in all_stats.items():
        print(f"{name:<10} {s['crowd_boxes']:>8} {s['tiny_boxes']:>8} "
              f"{s['images_without_boxes']:>14} {s['missing_dims']:>14}")
    print()
    print("Definitions:")
    print("  Crowd        - iscrowd=1 (annotated as group, not per-instance)")
    print("  Tiny         - bounding-box area < 32x32 px (1024 px2)")
    print("  Empty frames - images with 0 annotations")
    print("  Missing dims - images where PIL could not read width/height")
else:
    print("No stats available – run Section 3 first.")


---
## 6. Exploratory Data Analysis

The `scripts/eda/` package provides two modules:

| Module | Functions |
|--------|-----------|
| `eda.plots` | `plot_class_distribution`, `plot_bbox_size_distribution`, `plot_bbox_aspect_ratio`, `plot_objects_per_frame`, `plot_eda_dashboard` |
| `eda.visualizer` | `plot_annotated_samples`, `draw_annotations`, `plot_bbox_heatmap` |

All functions accept the `stats` dict from `compute_stats()` or a `DatasetSplit` object,  
and **return a `matplotlib.Figure`** – you decide whether to display inline or save to disk.

### 6a – Visualise annotated samples

`plot_annotated_samples(split, n=25, cols=5)` draws bounding boxes onto a grid of random images.  
This lets you visually verify that annotations are correct and understand the visual appearance of snowy scenes.

In [ ]:

# -- Section 6a: Annotated sample grid ---------------------------------------
# Displays n random training images with ground-truth bounding boxes drawn on them.
# Green boxes = class 0 (car). Each box shows the class name as a label.

from eda.visualizer import plot_annotated_samples

if "train_split" in dir() and train_split:
    try:
        fig = plot_annotated_samples(train_split, n=25, cols=5, seed=42)
        fig.savefig(FIGURES / "eda_annotated_samples_train.png", bbox_inches="tight", dpi=100)
        plt.show()
        print(f"Figure saved to {FIGURES / 'eda_annotated_samples_train.png'}")
    except FileNotFoundError as e:
        print(f"Images not found on disk: {e}")
        print("Make sure data/raw/images/ exists (download + extract NVD first).")
else:
    print("No train split loaded – run Section 3 first.")


### 6b – Distribution plots (EDA dashboard)

`plot_eda_dashboard(stats)` produces four subplots in one figure:
1. **Class distribution** – bar chart of total boxes per class
2. **Bounding-box size** – histograms of box width and height in pixels
3. **Aspect ratio** – histogram of width/height ratios with median line
4. **Objects per frame** – bar chart showing how many frames contain 0, 1, 2, ... boxes

These plots reveal:
- Whether the dataset is heavily single-class (which NVD likely is – mainly `car`)
- Typical vehicle sizes at this camera distance (useful for choosing anchor scales)
- Whether boxes are mostly wide (side-on vehicles) or tall (vehicles approaching head-on)

In [ ]:

# -- Section 6b: EDA dashboard (four distribution plots) --------------------
from eda.plots import plot_eda_dashboard

if all_stats and "train" in all_stats:
    fig = plot_eda_dashboard(all_stats["train"])
    fig.savefig(FIGURES / "eda_dashboard_train.png", bbox_inches="tight", dpi=100)
    plt.show()
    print(f"Figure saved to {FIGURES / 'eda_dashboard_train.png'}")
else:
    print("No train stats available – run Sections 3 and 4 first.")


### 6c – Bounding-box centre heatmap

`plot_bbox_heatmap(stats)` renders a 2D density map of all bounding-box centre positions,  
normalised to [0, 1] on both axes (so the heatmap is independent of image resolution).

**Why this is useful:**  
- Reveals camera bias – UAV/drone footage captures cars from an oblique aerial angle, so box centroids cluster differently than ground-level cameras
- If the heatmap shows a concentration in one corner it may indicate a labelling error or  
  a systematic recording issue
- Helps explain detection failures near the frame edges (fewer training examples)

In [ ]:

# -- Section 6c: Bounding-box centre heatmap ---------------------------------
from eda.visualizer import plot_bbox_heatmap

if all_stats and "train" in all_stats:
    fig = plot_bbox_heatmap(all_stats["train"])
    fig.savefig(FIGURES / "eda_bbox_heatmap_train.png", bbox_inches="tight", dpi=100)
    plt.show()
    print(f"Figure saved to {FIGURES / 'eda_bbox_heatmap_train.png'}")
else:
    print("No train stats available – run Sections 3 and 4 first.")


---
## 7. Snow Severity Labelling

To enable **per-condition analysis** (how does the model perform in heavy snow vs light snow?),
a sample of ~50 frames are manually labelled with one of three severity levels:
`light` / `medium` / `heavy`.

### How to run the interactive labeller

The script runs in the terminal. For each image it saves a preview to
`results/figures/label_preview.jpg` (open it in VS Code's image viewer) then prompts:

| Input | Label |
|-------|-------|
| `1` | light |
| `2` | medium |
| `3` | heavy |
| `s` | skip (image excluded from CSV) |
| `q` | quit and save progress |

```bash
python scripts/label_snow_severity.py \
    --images data/raw/images/ \
    --n      50 \
    --output data/snow_severity_sample.csv
```

You can **resume** a previous session with `--resume` – it skips files already labelled.

> **Headless note:** The script uses `input()` prompts instead of a GUI window so it works
> inside Docker containers or remote servers with no display. Each image is saved to
> `results/figures/label_preview.jpg` – VS Code auto-refreshes the file when you open it.

### Output format (`data/snow_severity_sample.csv`)

```
file_name,snow_severity
2022-12-02 Asjo 01_stabilized-frame0080.jpg,medium
2022-12-02 Asjo 01_stabilized-frame0081.jpg,heavy
...
```

This CSV is used later in Error Analysis to break down model performance by snow severity.


In [ ]:

# -- Section 7: Load and visualise snow severity labels -----------------------
SEVERITY_CSV = PROJECT_ROOT / "data" / "snow_severity_sample.csv"

if SEVERITY_CSV.exists():
    severity_df = pd.read_csv(SEVERITY_CSV)
    print(f"Loaded {len(severity_df)} labelled frames\n")
    print(severity_df["snow_severity"].value_counts().to_string())

    order = ["light", "medium", "heavy"]
    fig, ax = plt.subplots(figsize=(5, 3))
    counts = severity_df["snow_severity"].value_counts().reindex(order).fillna(0)
    ax.bar(order, counts, color=["#90caf9", "#42a5f5", "#1565c0"])
    ax.set_xlabel("Snow severity")
    ax.set_ylabel("Number of frames")
    ax.set_title("Manual snow severity distribution")
    for i, v in enumerate(counts):
        ax.text(i, v + 0.3, str(int(v)), ha="center", fontsize=10)
    fig.tight_layout()
    fig.savefig(FIGURES / "eda_snow_severity.png", bbox_inches="tight", dpi=100)
    plt.show()
else:
    print("No severity labels found.")
    print(f"Expected at: {SEVERITY_CSV}")
    print("\nRun from a terminal:")
    print("  python scripts/label_snow_severity.py \\")
    print("      --images data/raw/images/ \\")
    print("      --n 50 \\")
    print("      --output data/snow_severity_sample.csv")


---
## 8. Dataset Preparation

### 8a – Verify annotations visually (CLI)

`verify_annotations.py` draws bounding boxes onto sampled images and saves them as PNG files.  
Use this to confirm your label files and images are correctly aligned.

```bash
python scripts/verify_annotations.py \
    --data-yaml configs/data.yaml \
    --split     train \
    --n         10 \
    --output    results/figures/verify_train/
```

The script:
1. Reads `configs/data.yaml` to find the images and labels directories
2. Randomly samples `n` images from the requested split
3. Parses each matching `.txt` label file and draws boxes in green
4. Saves annotated images to `--output` folder

### 8b – Convert to COCO format (for DETR training)

DETR and other HuggingFace models expect **COCO JSON** format instead of YOLO `.txt` files.  
`prepare_coco_format.py` converts the NVD splits automatically.

```bash
python scripts/prepare_coco_format.py \
    --data-yaml configs/data.yaml \
    --output    data/processed/
```

**Output files:**
```
data/processed/
└── annotations/
    ├── instances_train.json
    ├── instances_val.json
    └── instances_test.json
```

Each JSON follows the [official COCO schema](https://cocodataset.org/#format-data):
```json
{
  "categories": [{"id": 1, "name": "car"}],
  "images":     [{"id": 1, "file_name": "...", "width": 1920, "height": 1080}],
  "annotations": [{"id": 1, "image_id": 1, "category_id": 1,
                   "bbox": [x, y, w, h], "area": 12345, "iscrowd": 0}]
}
```

> **Note:** COCO uses `[x, y, width, height]` (top-left corner + size), while YOLO uses  
> `[cx, cy, w, h]` normalised. The `coco_formatter.py` module handles this conversion.  
> Also, COCO category IDs are **1-indexed**; class_id 0 (car) maps to category_id 1.

In [ ]:

# -- Section 8a: Verify annotations via subprocess ---------------------------
# Equivalent to running the CLI from a terminal.

import subprocess

result = subprocess.run(
    [sys.executable, "scripts/verify_annotations.py",
     "--data-yaml", "configs/data.yaml",
     "--split",     "train",
     "--n",         "10",
     "--output",    "results/figures/verify_train/"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)


In [ ]:

# -- Section 8b: Convert YOLO splits to COCO JSON ----------------------------
# Only needed if you are training DETR or another HuggingFace model.
# Skip if you are training YOLOv9 only.

result = subprocess.run(
    [sys.executable, "scripts/prepare_coco_format.py",
     "--data-yaml", "configs/data.yaml",
     "--output",    "data/processed/"],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

# Check the output
coco_dir = PROJECT_ROOT / "data" / "processed" / "annotations"
if coco_dir.exists():
    for f in sorted(coco_dir.glob("*.json")):
        import json
        with open(f) as jf:
            d = json.load(jf)
        print(f"  {f.name}: {len(d['images'])} images, "
              f"{len(d['annotations'])} annotations, "
              f"{len(d['categories'])} categories")


---
## 9. Model Training – YOLOv9

We use **YOLOv9 via `ultralytics`** (Option A). The ultralytics library provides a unified  
Python API for training, validation, and inference across all YOLO variants.

### Why YOLOv9?
- **Programmable Gradient Information (PGI)** – a new training strategy that maintains  
  complete gradient information through auxiliary branches, addressing the information  
  bottleneck in deep networks
- **Generalised ELAN (GELAN)** – an efficient feature aggregation network that achieves  
  better parameter utilisation than previous YOLO architectures
- Strong COCO performance as a starting point for fine-tuning

### Training strategy

| Stage | Configuration | Purpose |
|-------|--------------|---------|
| Zero-shot | No fine-tuning | Anchor point – establishes how bad COCO-only performance is on NVD |
| Fine-tuned | `freeze=10`, `epochs=50`, `lr0=0.01` | Adapt to NVD domain |
| + Snow aug | Add `albumentations` snow transform | Test augmentation hypothesis |
| + Full aug | Snow + random brightness/contrast | Maximise augmentation coverage |

### The `configs/data.yaml` file

```yaml
path: ../data/raw        # dataset root (relative to configs/)
train: train.txt         # paths to training images
val:   val.txt
test:  test.txt
nc: 1                    # number of classes
names:
  0: car
```

This file is passed directly to `model.train(data="configs/data.yaml", ...)`.  
No conversion step is needed – ultralytics reads YOLO `.txt` label files natively.

In [ ]:

# -- Section 9a: Zero-shot baseline (COCO-pretrained, no fine-tuning) --------
# Run COCO-pretrained YOLOv9 on the NVD val set WITHOUT any fine-tuning.
# This gives you the "before" anchor for your fine-tuning improvement story.
#
# Uncomment and run when you have the dataset downloaded.

# from ultralytics import YOLO
#
# model = YOLO("yolov9c.pt")   # downloads COCO weights automatically on first run
# zero_shot_results = model.val(
#     data="configs/data.yaml",
#     split="val",
#     project="results",
#     name="yolov9_zero_shot",
#     save_json=True,
# )
#
# print(f"mAP@0.5:      {zero_shot_results.box.map50:.4f}")
# print(f"mAP@0.5:0.95: {zero_shot_results.box.map:.4f}")
# print(f"Precision:    {zero_shot_results.box.mp:.4f}")
# print(f"Recall:       {zero_shot_results.box.mr:.4f}")

print("Zero-shot baseline cell – uncomment and run after downloading NVD.")


In [ ]:

# -- Section 9b: Fine-tune YOLOv9 on NVD ------------------------------------
# Key parameters:
#   freeze=10     - freeze the first 10 layers (backbone). The head is trained from scratch.
#   epochs=50     - starting point; increase if val mAP is still improving at epoch 50
#   batch=16      - reduce to 8 if you run out of VRAM (6 GB GPU)
#   imgsz=640     - standard YOLOv9 input resolution
#   lr0=0.01      - initial learning rate; cosine decay by default
#   project/name  - organises outputs into results/yolov9_nvd/
#
# Uncomment and run when you have the dataset downloaded.

# from ultralytics import YOLO
#
# model = YOLO("yolov9c.pt")
# train_results = model.train(
#     data="configs/data.yaml",
#     epochs=50,
#     batch=16,
#     imgsz=640,
#     lr0=0.01,
#     freeze=10,
#     project="results",
#     name="yolov9_nvd",
#     exist_ok=True,
# )
#
# # Save the best checkpoint to models/
# import shutil
# best_ckpt = Path("results/yolov9_nvd/weights/best.pt")
# if best_ckpt.exists():
#     shutil.copy(best_ckpt, MODELS_DIR / "yolov9_nvd_best.pt")
#     print(f"Best checkpoint saved to {MODELS_DIR / 'yolov9_nvd_best.pt'}")

print("Fine-tuning cell – uncomment and run after downloading NVD.")


In [ ]:

# -- Section 9c: Snow augmentation ablation ----------------------------------
# To test the hypothesis that synthetic snow augmentation helps:
# 1. Train baseline (no augmentation beyond ultralytics defaults)
# 2. Train with albumentations RandomSnow added as a custom augmentation
# 3. Train with full augmentation (snow + brightness/contrast + blur)
#
# ultralytics supports custom augmentation via the `augment` callback or via
# a custom albumentations transform passed in hyp.yaml. A simple approach is:

# import albumentations as A
# from ultralytics.utils import LOGGER
#
# # Define the snow augmentation pipeline
# snow_transform = A.Compose([
#     A.RandomSnow(snow_point_lower=0.1, snow_point_upper=0.3,
#                  brightness_coeff=2.5, p=0.4),
#     A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.3),
# ])
#
# # Then pass this to model.train via hyp overrides or a custom callback.
# # See the EXPERIMENTS.md for run names and results once training completes.

print("Snow augmentation cell – see EXPERIMENTS.md for ablation run configs.")
print()
print("Run names to use in wandb:")
print("  yolov9_nvd_no_aug    – baseline, no extra augmentation")
print("  yolov9_nvd_snow_aug  – + RandomSnow (p=0.4)")
print("  yolov9_nvd_full_aug  – + RandomSnow + BrightnessContrast + Blur")


---
## 10. Model Training – DETR

**DETR (DEtection TRansformer)** is a transformer-based object detector from Facebook AI.  
Unlike YOLO, it treats object detection as a set-prediction problem using a bipartite  
matching loss – no anchor boxes, no NMS post-processing.

We use **`facebook/detr-resnet-50`** from HuggingFace as the pretrained starting point.

### Key differences from YOLOv9

| Aspect | YOLOv9 | DETR |
|--------|--------|------|
| Architecture | CNN + detection head | CNN backbone + Transformer encoder-decoder |
| Post-processing | NMS (built-in) | None (Hungarian matching) |
| Training data format | YOLO `.txt` | COCO JSON |
| Speed (inference) | ~30–100 FPS | ~10–30 FPS |
| Fine-tuning API | `ultralytics` Python API | HuggingFace `Trainer` |

### COCO format requirement

DETR requires COCO JSON annotations. Run Section 8b (`prepare_coco_format.py`)  
before training DETR.

In [ ]:

# -- Section 10: Fine-tune DETR with HuggingFace Transformers ----------------
# Prerequisites:
#   1. Run Section 8b to create data/processed/annotations/instances_*.json
#   2. pip install transformers accelerate datasets  (already in requirements.txt)
#
# Training script: scripts/train_detr.py (to be created in Phase 2)
# See: https://huggingface.co/docs/transformers/model_doc/detr

# from transformers import DetrForObjectDetection, DetrImageProcessor
# from torch.utils.data import DataLoader
#
# processor = DetrImageProcessor.from_pretrained("facebook/detr-resnet-50")
# model = DetrForObjectDetection.from_pretrained(
#     "facebook/detr-resnet-50",
#     num_labels=1,          # 1 class: car
#     ignore_mismatched_sizes=True,
# )
#
# # See scripts/train_detr.py for the full training loop with:
# #   - CocoDetection dataset wrapper
# #   - custom collate_fn for variable-length annotation lists
# #   - Hungarian matching loss (built into DETR forward pass)
# #   - Trainer API from HuggingFace accelerate

print("DETR training cell – see scripts/train_detr.py (Phase 2 deliverable).")
print("Requires COCO JSON annotations from Section 8b first.")


---
## 11. Evaluation & Results

### Metrics

| Metric | Description |
|--------|-------------|
| **mAP@0.5** | Mean Average Precision at IoU threshold 0.5 – the standard detection metric |
| **mAP@0.5:0.95** | mAP averaged over IoU thresholds 0.5–0.95 (stricter, from COCO benchmark) |
| **Precision** | Of all predicted boxes, what fraction were correct? |
| **Recall** | Of all ground-truth boxes, what fraction did we find? |
| **F1** | Harmonic mean of precision and recall |
| **FPS** | Inference frames per second (model throughput) |

### What counts as a "correct" detection?

A predicted box is a **True Positive** if:
- Its class prediction matches the ground-truth class, AND
- Its IoU with the closest ground-truth box >= threshold (0.5 for mAP@0.5)

Intersection over Union (IoU):  
$$\text{IoU} = \frac{\text{Area of Overlap}}{\text{Area of Union}}$$

### Logging results to `EXPERIMENTS.md`

After each training run, add a row to `EXPERIMENTS.md`:

```markdown
| 1 | 2026-04-18 | YOLOv9c | freeze=10, epochs=50 | 0.82 | 0.61 | 0.88 | 0.79 | 47 | run_xyz | First fine-tune |
```

In [ ]:

# -- Section 11a: Evaluate fine-tuned YOLOv9 on the test set -----------------
# Run this cell after training completes and the checkpoint exists at
# models/yolov9_nvd_best.pt

# from ultralytics import YOLO
#
# model = YOLO(str(MODELS_DIR / "yolov9_nvd_best.pt"))
# test_results = model.val(
#     data="configs/data.yaml",
#     split="test",
#     project="results",
#     name="yolov9_nvd_test_eval",
# )
#
# print(f"Test mAP@0.5:      {test_results.box.map50:.4f}")
# print(f"Test mAP@0.5:0.95: {test_results.box.map:.4f}")
# print(f"Test Precision:    {test_results.box.mp:.4f}")
# print(f"Test Recall:       {test_results.box.mr:.4f}")

print("Evaluation cell – uncomment after training and saving the checkpoint.")


In [ ]:

# -- Section 11b: Model comparison table -------------------------------------
# Fill in metrics after all experiments are complete.
# Update EXPERIMENTS.md at the same time.

results_data = {
    "Model":            ["YOLOv9 (zero-shot)", "YOLOv9 fine-tuned",
                         "YOLOv9 + snow aug",  "YOLOv9 + full aug",
                         "DETR (ResNet-50)",   "Faster R-CNN"],
    "Pretrain":         ["COCO"] * 6,
    "Fine-tuned":       ["N", "Y", "Y", "Y", "Y", "Y"],
    "mAP@0.5":          ["-"] * 6,
    "mAP@0.5:0.95":     ["-"] * 6,
    "Precision":        ["-"] * 6,
    "Recall":           ["-"] * 6,
    "FPS":              ["-"] * 6,
}

results_df = pd.DataFrame(results_data).set_index("Model")
display(results_df)
print("\nFill in the '-' values from EXPERIMENTS.md after each training run.")


---
## 12. Error Analysis

Error analysis is a systematic way to understand **where and why** the model fails.  
This is a key part of any deep learning project report.

### Three failure categories

| Category | Description | Example |
|----------|-------------|---------|
| **False Negative (FN)** | Model missed a vehicle that is there | Car obscured by snow/fog |
| **False Positive (FP)** | Model detected a vehicle that is not there | Snow pile or shadow |
| **Poor Localisation** | Detected the vehicle but the box is badly placed | IoU > 0 but < 0.5 |

### Process

1. Run inference on the test set and collect all detections
2. Match predictions to ground truth (Hungarian matching at IoU >= 0.5)
3. Sort unmatched GTs by confidence score → False Negatives
4. Sort unmatched predictions by confidence → False Positives
5. Visualise the worst 20 FNs, 10 FPs, 10 poor localisation cases
6. Annotate each failure image with the snow severity label (from `snow_severity_sample.csv`)
7. Look for patterns: do FNs cluster in heavy snow? Are FPs near bright reflections?

### Per-condition breakdown (using severity labels)

After computing predictions, join with the severity CSV to compute mAP per severity level:
```python
# Pseudo-code
for severity in ["light", "medium", "heavy"]:
    subset_images = severity_df[severity_df["snow_severity"] == severity]["file_name"].tolist()
    # Compute mAP on this subset
```
This reveals whether the model degrades disproportionately in heavy snow – the core research question.

In [ ]:

# -- Section 12: Run inference and visualise detections ----------------------
# After training, run inference on a sample of test images and display results.
# Uncomment after the checkpoint exists.

# from ultralytics import YOLO
# import cv2
#
# model = YOLO(str(MODELS_DIR / "yolov9_nvd_best.pt"))
#
# # Get a sample of test images
# test_txt = DATA_RAW / "test.txt"
# test_image_paths = [line.strip() for line in test_txt.read_text().splitlines()
#                     if line.strip()][:9]  # first 9
#
# fig, axes = plt.subplots(3, 3, figsize=(15, 10))
# for ax, img_path in zip(axes.flatten(), test_image_paths):
#     full_path = DATA_RAW / img_path if not Path(img_path).is_absolute() else Path(img_path)
#     results = model(str(full_path), verbose=False)
#     annotated = results[0].plot()  # returns BGR numpy array
#     ax.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
#     ax.axis("off")
#
# fig.suptitle("YOLOv9 fine-tuned – test set predictions", fontsize=13)
# fig.tight_layout()
# fig.savefig(FIGURES / "yolov9_test_predictions.png", bbox_inches="tight", dpi=100)
# plt.show()

print("Inference visualisation cell – uncomment after training completes.")


---
## Summary: How to Use This Codebase

### Quick reference – most important commands

```bash
# 1. Inspect the dataset (print stats for all splits)
python scripts/inspect_dataset.py --data-yaml configs/data.yaml

# 2. Visually verify annotations (draw boxes on 10 sample images)
python scripts/verify_annotations.py --data-yaml configs/data.yaml --split train --n 10 --output results/figures/verify_train/

# 3. Label ~50 frames by snow severity interactively
python scripts/label_snow_severity.py --images data/raw/images/ --n 50 --output data/snow_severity_sample.csv

# 4. Convert to COCO format (needed for DETR)
python scripts/prepare_coco_format.py --data-yaml configs/data.yaml --output data/processed/
```

### Quick reference – Python API

```python
import sys
sys.path.insert(0, "scripts/")

from data_utils import load_yolo_split_from_txt, compute_stats
from data_utils.reporting import print_stats
from eda.plots import plot_eda_dashboard
from eda.visualizer import plot_annotated_samples, plot_bbox_heatmap

# Load a split
split = load_yolo_split_from_txt("data/raw/train.txt", "data/raw", "train", ["car"])

# Compute and print stats
stats = compute_stats(split)
print_stats(stats)

# EDA plots
fig = plot_eda_dashboard(stats)       # 4-panel distribution plot
fig = plot_annotated_samples(split)   # grid of annotated images
fig = plot_bbox_heatmap(stats)        # 2D heatmap of box centres
```

### Phase checklist

| Phase | Status | Key deliverable |
|-------|--------|----------------|
| Phase 0 – Setup | ✅ Done | `requirements.txt`, `configs/`, repo structure |
| Phase 1 – Data & EDA | 🔄 In progress | EDA cells in `main.ipynb`, `snow_severity_sample.csv` |
| Phase 2 – Baselines | ⬜ Pending | Zero-shot eval, YOLOv9 fine-tune, DETR fine-tune |
| Phase 3 – Augmentation | ⬜ Pending | 3 ablation runs, `EXPERIMENTS.md` filled |
| Phase 4 – Analysis | ⬜ Pending | Error analysis grids, per-severity breakdown |
| Phase 5 – Report | ⬜ Pending | Final paper in IEEE format |